In [ ]:

%pip install -q -U crewai crewai-tools


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [ ]:

from google.colab import userdata
import os

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Add GEMINI_API_KEY in Colab Secrets.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("API key loaded successfully.")


API key loaded successfully.


In [ ]:

from crewai import Agent, Task, Crew, Process, LLM
from crewai.flow.flow import Flow, start, listen, and_
from crewai_tools import ScrapeWebsiteTool
from pydantic import BaseModel, Field
from pathlib import Path
from urllib.parse import urlparse
import json

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.4
)

OUTPUT_DIR = Path("/content/content_repurposing_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup completed.")


Setup completed.


In [ ]:

from pathlib import Path
from crewai_tools import ScrapeWebsiteTool

def read_article(source):
    source = source.strip()

    if not source:
        raise ValueError(
            "Please provide article text, a file path, or a URL."
        )

    # 1. Read article from a URL
    if source.startswith(("http://", "https://")):
        try:
            scraper = ScrapeWebsiteTool(website_url=source)
            content = scraper.run()

            if not content or not str(content).strip():
                raise ValueError(
                    "Could not extract readable content from this URL."
                )

            return str(content).strip()

        except Exception as e:
            raise ValueError(f"Failed to scrape article URL: {e}") from e

    # 2. Read article from a file path
    try:
        path = Path(source)
        if path.is_file():
            content = path.read_text(encoding="utf-8").strip()

            if not content:
                raise ValueError("The article file is empty.")

            return content
    except (OSError, ValueError) as e:
        if isinstance(e, ValueError) and str(e) == "The article file is empty.":
            raise

    # 3. Otherwise, treat the input as article text
    if len(source) < 50:
        raise ValueError(
            "Input is too short to be an article. "
            "Provide article text, a valid file path, or a URL."
        )

    return source


print("read_article() updated successfully.")


read_article() updated successfully.


In [ ]:

def create_linkedin_crew():
    agent = Agent(
        role="LinkedIn Content Strategist",
        goal="Create a compelling LinkedIn post of approximately 150 words.",
        backstory="You specialize in professional, engaging social content.",
        llm=llm,
        verbose=True
    )

    task = Task(
        description=(
            "Using these article points:\n{article_points}\n\n"
            "Write a LinkedIn post of approximately 150 words. Start with a "
            "strong hook, use readable paragraphs, preserve the facts, and "
            "finish with a relevant question or call to action."
        ),
        expected_output="One polished LinkedIn post of approximately 150 words.",
        agent=agent
    )

    return Crew(
        agents=[agent], tasks=[task],
        process=Process.sequential, verbose=True
    )


def create_twitter_crew():
    agent = Agent(
        role="Twitter/X Thread Writer",
        goal="Create an engaging six-tweet thread from the article.",
        backstory="You turn complex ideas into concise, readable tweets.",
        llm=llm,
        verbose=True
    )

    task = Task(
        description=(
            "Using these article points:\n{article_points}\n\n"
            "Write exactly six numbered tweets. Tweet 1 must hook the reader. "
            "Arrange the ideas logically and finish with a useful takeaway. "
            "Keep tweets concise and do not invent facts."
        ),
        expected_output="Exactly six numbered tweets, from 1/6 to 6/6.",
        agent=agent
    )

    return Crew(
        agents=[agent], tasks=[task],
        process=Process.sequential, verbose=True
    )


def create_newsletter_crew():
    agent = Agent(
        role="Newsletter Writer",
        goal="Create a newsletter introduction and three key bullet points.",
        backstory="You write concise newsletters that highlight useful insights.",
        llm=llm,
        verbose=True
    )

    task = Task(
        description=(
            "Using these article points:\n{article_points}\n\n"
            "Write a short newsletter introduction followed by exactly three "
            "informative bullet points. Preserve the source meaning and do "
            "not invent facts."
        ),
        expected_output="A newsletter introduction and exactly three bullets.",
        agent=agent
    )

    return Crew(
        agents=[agent], tasks=[task],
        process=Process.sequential, verbose=True
    )


print("All three Crew builders created.")


All three Crew builders created.


In [ ]:
from crewai.flow.flow import Flow, start, listen, and_
from pydantic import BaseModel, Field

class RepurposingState(BaseModel):
    source: str = ""
    article_text: str = ""
    article_points: str = ""
    linkedin_post: str = ""
    twitter_thread: str = ""
    newsletter: str = ""
    output_files: dict = Field(default_factory=dict)

In [ ]:

class ContentRepurposingFlow(Flow[RepurposingState]):

    @start()
    def load_article(self):
        self.state.article_text = read_article(self.state.source)

        if not self.state.article_text.strip():
            raise ValueError("The article contains no readable text.")

        print("Article loaded.")
        print("Characters:", len(self.state.article_text))

    @listen(load_article)
    def extract_points(self):
        agent = Agent(
            role="Article Insight Extractor",
            goal="Extract the central ideas and important supporting details.",
            backstory="You summarize source material accurately without inventing facts.",
            llm=llm,
            verbose=True
        )

        task = Task(
            description=(
                "Extract the key ideas from this article:\n\n"
                "{article}\n\n"
                "Return a structured list of the main argument, important "
                "supporting details, and conclusion. Do not add outside facts."
            ).format(article=self.state.article_text),
            expected_output="A concise list of key ideas and the main conclusion.",
            agent=agent
        )

        result = Crew(
            agents=[agent],
            tasks=[task],
            process=Process.sequential,
            verbose=True
        ).kickoff()

        self.state.article_points = str(result)
        print("Key points extracted.")

    @listen(extract_points)
    def create_linkedin(self):
        result = create_linkedin_crew().kickoff(
            inputs={"article_points": self.state.article_points}
        )
        self.state.linkedin_post = str(result)
        print("LinkedIn Crew completed.")

    @listen(extract_points)
    def create_twitter(self):
        result = create_twitter_crew().kickoff(
            inputs={"article_points": self.state.article_points}
        )
        self.state.twitter_thread = str(result)
        print("Twitter Crew completed.")

    @listen(extract_points)
    def create_newsletter(self):
        result = create_newsletter_crew().kickoff(
            inputs={"article_points": self.state.article_points}
        )
        self.state.newsletter = str(result)
        print("Newsletter Crew completed.")

    @listen(and_(create_linkedin, create_twitter, create_newsletter))
    def publish_pack(self):
        outputs = {
            "linkedin_post.md": self.state.linkedin_post,
            "twitter_thread.md": self.state.twitter_thread,
            "newsletter.md": self.state.newsletter
        }

        for filename, content in outputs.items():
            path = OUTPUT_DIR / filename
            path.write_text(content, encoding="utf-8")
            self.state.output_files[filename] = str(path)

        summary = {
            "source": self.state.source,
            "article_points": self.state.article_points,
            "output_files": self.state.output_files
        }

        summary_path = OUTPUT_DIR / "summary.json"
        summary_path.write_text(
            json.dumps(summary, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

        print("\nAll three branches completed.")
        print("Saved files:", self.state.output_files)


In [ ]:

from google.colab import files
from urllib.request import Request, urlopen

choice = input(
    "Choose article input: 1 = Upload TXT file, 2 = Enter URL: "
).strip()

sample_article = ""

if choice == "1":
    uploaded = files.upload()

    for filename, content in uploaded.items():
        if filename.lower().endswith(".txt"):
            sample_article = content.decode("utf-8").strip()
            break

    if not sample_article:
        raise ValueError("Please upload a non-empty .txt article.")

elif choice == "2":
    article_url = input("Enter the article URL: ").strip()

    if not article_url.startswith(("https://", "http://")):
        raise ValueError("Please enter a valid HTTP/HTTPS URL.")

    try:
        request = Request(
            article_url,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        with urlopen(request, timeout=20) as response:
            html = response.read().decode("utf-8", errors="replace")

        from bs4 import BeautifulSoup

        soup = BeautifulSoup(html, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        sample_article = soup.get_text(separator=" ", strip=True)

        if not sample_article:
            raise ValueError("No readable article text found at this URL.")

    except Exception as e:
        raise ValueError(f"Could not load article: {e}")

else:
    raise ValueError("Choose 1 for TXT or 2 for URL.")

print(f"\nArticle loaded: {len(sample_article)} characters")

flow = ContentRepurposingFlow()
flow.state.source = sample_article
flow.kickoff()


╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ContentRepurposingFlow                                                                                   │
│  ID: b0b05f74-22ba-44ea-ad24-68d321f657bd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ContentRepurposingFlow                                                                                   │
│  ID: b0b05f74-22ba-44ea-ad24-68d321f657bd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Article loaded.
Characters: 672


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_article                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: extract_points                                                                                         │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


All three branches completed.
Saved files: {'linkedin_post.md': '/content/content_repurposing_output/linkedin_post.md', 'twitter_thread.md': '/content/content_repurposing_output/twitter_thread.md', 'newsletter.md': '/content/content_repurposing_output/newsletter.md'}


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: create_twitter                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: eab1e0b7-efb7-4fed-a030-c23fcebf74d1                                                                       │
│  Final Output: 1/6 The future of education isn't about banning artificial intelligence. The most effective      │
│  approach combines AI assistance with your own independent thinking. Let's dive in. 🧵                          │
│                                                                                                                 │
│  2/6 AI is a powerful study partner. It can help you brainstorm ideas, break down difficult concepts, practice  │
│  coding, and give you instant feedback on your work.                                                            │
│                                                                                                                 │
│  3/6 But remember: AI-generated information isn't infallible. It can and does contain errors. Blindly copying   │
│  and pasting answers stunts your growth and leads to mistakes.                                                  │
│                                                                                                                 │
│  4/6 Instead of taking answers at face value, use AI to explore alternatives and identify gaps in your          │
│  understanding. Let it guide you, not replace you.                                                              │
│                                                                                                                 │
│  5/6 Always verify important claims. Cross-reference AI results using reliable sources and test your knowledge  │
│  with practical exercises to ensure you actually learned the material.                                          │
│                                                                                                                 │
│  6/6 Used responsibly, AI supports learning without replacing the effort needed to build real skills. AI        │
│  should be a springboard for your own critical thinking, not a substitute for it.                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: publish_pack                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ContentRepurposingFlow                                                                                   │
│  ID: b0b05f74-22ba-44ea-ad24-68d321f657bd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:

for filename in [
    "linkedin_post.md",
    "twitter_thread.md",
    "newsletter.md"
]:
    path = OUTPUT_DIR / filename

    print(f"\n{'=' * 15} {filename} {'=' * 15}")
    print(path.read_text(encoding="utf-8"))

print("\nOutput directory:", OUTPUT_DIR)
print("Summary:", OUTPUT_DIR / "summary.json")



=============== linkedin_post.md ===============
AI is transforming education, but the most effective approach isn't avoidance—it's combination. True learning in the age of artificial intelligence happens when we pair smart tech with independent thinking. 

AI tools are incredible force multipliers. They help students brainstorm ideas, break down complex concepts, practice coding, and receive instant feedback. They allow learners to explore alternative perspectives and spot gaps in their understanding.

However, convenience shouldn't replace critical thinking. AI can—and does—generate errors. Blindly copying answers builds no real skills. That’s why students must verify important claims, cross-check results with reliable sources, and test insights through practical exercises. 

Used responsibly, AI doesn’t do the heavy lifting for us. Instead, it supports our learning journey while preserving the essential effort needed to master new skills. 

How are you balancing AI tools with criti

In [ ]:

from google.colab import files

for filename in [
    "linkedin_post.md",
    "twitter_thread.md",
    "newsletter.md",
    "summary.json"
]:
    path = OUTPUT_DIR / filename

    if path.exists():
        files.download(str(path))
        print(f"Downloading: {filename}")
    else:
        print(f"File not found: {filename}")
